# 🧬 Project 5 — Semantic Correspondence with Visual Foundation Models
**Advanced Machine Learning 2025/2026**

Questo notebook contiene la pipeline completa e modulare per il progetto: dall'analisi della baseline (Stage 1) all'addestramento efficiente (Stage 2) fino alle estensioni personalizzate (Stage 3-4).

**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

## 📦 0. Setup & Installazione

In [ ]:
# Collega Google Drive per salvare i pesi in modo persistente
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone Repo e installazione dipendenze
!git clone -b osagie5 https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!pip install -r requirements.txt -q
!pip install gradio -q

# Download dataset SPair-71k (Rapido)
!python dataloaders/download_spair.py --root ./data

## 🔍 1. Stage 1: Analisi Baseline (DINOv2 puro)
In questa fase verifichiamo come si comporta il modello pre-addestrato senza fine-tuning.

In [ ]:
# Valutazione Baseline (Layer 11 - Ultimo)
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --no_adaptive_win

In [ ]:
# 🎯 Analisi Bonus: Layer-wise Comparison
# Confrontiamo le performance estraendo feature da diverse profondità del backbone
for layer in [4, 8, 11]:
    print(f"\n--- Valutazione Layer {layer} ---")
    !python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --layer {layer} --no_adaptive_win

## 🚀 2. Stage 2: Fine-Tuning Efficiente (LoRA + Curriculum)
Addestriamo i parametri LoRA per adattare DINOv2 al compito di corrispondenza semantica.

In [ ]:
# Questa run dura circa 2.5 ore su T4.
# Utilizziamo: LoRA, Curriculum Learning e Adaptive Soft-Argmax.
!python train.py \
    --dataset_root ./data/SPair-71k \
    --backbone dinov2_vitb14 \
    --epochs 5 \
    --curriculum_epochs 2 \
    --aw_min_radius 2 \
    --aw_max_radius 7 \
    --output_dir ./checkpoints \
    --backup_dir /content/drive/MyDrive/AML/Checkpoints

## 🏁 3. Stage 3: Valutazione Finale
Confrontiamo i risultati del modello addestrato rispetto alla baseline iniziale.

In [ ]:
# Valutazione completa con analisi per categoria
!python evaluate.py \
    --dataset_root ./data/SPair-71k \
    --checkpoint /content/drive/MyDrive/AML/Checkpoints/best.pth

## 🌟 4. Stage 4: Estensioni & Demo

In [ ]:
# 🎮 Demo Interattiva con Gradio
# Carica due immagini e clicca per vedere il match predetto dal modello!
!python demo_gradio.py --checkpoint /content/drive/MyDrive/AML/Checkpoints/best.pth --share